# 147 — Model Merging (SLERP)

**What you will learn:**

- Why LERP loses vector magnitude and why SLERP travels the sphere surface instead
- The omega formula: `omega = arccos(v0 . v1 / (|v0| * |v1|))` and what each term means
- How the `t` parameter sweeps from model A (t=0) to model B (t=1) and how to search for the optimal value
- The three rules for state-dict merging: shape match, floating-point check, and the fallback when either fails
- Exercise: implement an optimal-t search over mock task scores
- Exercise: implement selective merging that skips divergent layers based on cosine similarity
- Live Part B: merge `SmolLM2-135M-Instruct` and `SmolLM2-360M-Instruct` on CPU and compare outputs

**Source code:** `examples/147-model-merging/`

**Models used:** `HuggingFaceTB/SmolLM2-135M-Instruct` and `HuggingFaceTB/SmolLM2-360M-Instruct`

**Structure:**
- **Part A** — CPU-safe concept demonstrations. All cells run on CPU with tiny tensors. No model downloads required.
- **Part B** — Full model merge. Requires approximately 2 GB free RAM and an internet connection for the first download (~300 MB total).

In [ ]:
%pip install -q transformers torch psutil

## Part A — CPU-Safe Concept Demonstrations

All cells in Part A run on CPU with tiny synthetic tensors. No model downloads needed. Each cell is self-contained and can be run independently.

### A1 — When to Merge vs Fine-tune

Model merging is sometimes called "free-data fine-tuning": you combine the skills of two already-trained models without needing a single labeled example or GPU training run. The trade-off is that quality is not guaranteed — you are hoping the two weight spaces are compatible.

Use this table to decide which approach fits your situation:

| Approach | Compute Cost | Data Needed | Quality Guarantee | Best When |
|---|---|---|---|---|
| Fine-tuning | GPU-hours | Thousands of examples | High — directly optimized | You have labeled data and a clear task |
| SLERP merge | CPU minutes | Zero examples | Moderate — depends on compatibility | Combining complementary skills from related models |
| Prompt engineering | Zero | Zero | Low — surface-level only | Quick iteration, no training budget |

The key insight is that merging works best when model A and model B were fine-tuned from the **same base model**. In that case, their weight tensors are close in weight space and SLERP can blend them smoothly. When the two models diverge significantly (different architectures, different base checkpoints), the merged model often degrades.

### A2 — Model Merging Methods

Several merging strategies exist. SLERP is the default starting point; the others address specific failure modes.

| Method | Full Name | Key Idea | When to Use |
|---|---|---|---|
| LERP | Linear interpolation | Weighted average of raw weight values | Quick baseline; loses magnitude at midpoints |
| SLERP | Spherical linear interpolation | Travels an arc on the weight-space sphere, preserving magnitude | Default choice for two-model merges |
| TIES | Trim Interpolate Sign | Resolves sign conflicts between delta weights before averaging | Merging many fine-tuned models from the same base |
| DARE | Drop And REscale | Randomly drops small delta weights before merging to reduce task interference | Large model families with many fine-tunes |

**This notebook focuses on SLERP.** Understanding SLERP's geometry gives you the intuition needed to reason about all the other methods.

### A3 — Architecture Compatibility Requirements

For merging to work, both models must have **identical architecture**: same layer names and same tensor shapes at every key. When shapes differ, that layer is kept from model A unchanged.

This means merging a 135M and a 360M model will skip most embedding and head layers (their vocabulary or hidden dimensions differ) but can still SLERP-merge shared layers if they happen to share the same shape.

```
Model A state dict               Model B state dict
-------------------------------  -------------------------------
embed_tokens.weight [V, 512]     embed_tokens.weight [V, 768]   <-- shape mismatch, kept from A
layer.0.attn.q_proj [512, 512]   layer.0.attn.q_proj [512, 512] <-- shapes match, SLERP merged
layer.0.attn.v_proj [512, 512]   layer.0.attn.v_proj [512, 512] <-- shapes match, SLERP merged
lm_head.weight [V, 512]          lm_head.weight [V, 768]        <-- shape mismatch, kept from A
```

**Practical rule:** always check `n_merged / n_total` after a merge. If fewer than 30% of layers merged, the two models are too architecturally different and the merge result will behave almost identically to model A.

### A4 — The Magnitude Problem: LERP vs SLERP

LERP computes `(1 - t) * v0 + t * v1`, which travels through the **interior** of the unit sphere. At t=0.5, two orthogonal unit vectors average to a vector with magnitude `sqrt(2)/2 = 0.707`. That is a 29% magnitude reduction — the merged weights are systematically smaller than either parent.

```
         v1 (magnitude = 1.0)
          |
          |  <- SLERP arc: stays ON the sphere surface (magnitude = 1.0 throughout)
         -*-
        / | \
       /  |  \
      /   +   \  <- LERP midpoint: inside the sphere (magnitude = 0.707 at t=0.5)
     /         \
   v0 ---------- (unit circle cross-section)
```

SLERP fixes this by traveling the arc between the two vectors. The formula is:

```
SLERP(t, v0, v1) = sin((1-t) * omega) / sin(omega) * v0
                 + sin(t * omega)      / sin(omega) * v1
```

where `omega = arccos(v0 . v1 / (|v0| * |v1|))` is the angle between the two vectors.

**Breaking down each term:**
- `omega` — the total arc angle from v0 to v1 on the sphere
- `(1-t) * omega` — the remaining angle from the interpolated point back to v0
- `t * omega` — the angle from v0 to the interpolated point (toward v1)
- `sin(omega)` — normalization factor that ensures the weights sum to produce a unit vector

When `omega` is very small (vectors nearly parallel), `sin(omega) ~ 0` causes division instability. The implementation falls back to LERP in that case, which is safe because the two vectors are already close.

In [ ]:
# A5 — SLERP magnitude demo: orthogonal vectors
import torch

def slerp(t: float, v0: torch.Tensor, v1: torch.Tensor) -> torch.Tensor:
    """Spherical linear interpolation between v0 and v1.

    Args:
        t: interpolation parameter in [0, 1]. t=0 returns v0, t=1 returns v1.
        v0: starting tensor.
        v1: ending tensor. Must be same shape as v0.

    Returns:
        Interpolated tensor with the same dtype as v0.
    """
    v0_f = v0.float()
    v1_f = v1.float()
    dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-4:
        # Fallback to LERP when vectors are nearly parallel
        return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
    so = torch.sin(omega)
    return (
        torch.sin((1 - t) * omega) / so * v0_f
        + torch.sin(t * omega) / so * v1_f
    ).to(v0.dtype)


# Two orthogonal unit vectors in 4D: the angle between them is exactly 90 degrees
v0 = torch.tensor([1.0, 0.0, 0.0, 0.0])
v1 = torch.tensor([0.0, 1.0, 0.0, 0.0])

print(f"{'t':>5}  {'SLERP result':>30}  {'SLERP |v|':>10}  {'LERP |v|':>10}")
print("-" * 65)
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    slerp_v = slerp(t, v0, v1)
    lerp_v = (1 - t) * v0 + t * v1
    slerp_vals = "[" + ", ".join(f"{x:.3f}" for x in slerp_v.tolist()) + "]"
    print(f"{t:>5.2f}  {slerp_vals:>30}  {slerp_v.norm().item():>10.4f}  {lerp_v.norm().item():>10.4f}")

print()
print("Key observation:")
print("  SLERP magnitude stays at ~1.000 throughout the interpolation.")
print("  LERP magnitude dips to 0.7071 at t=0.5 (sqrt(2)/2 for orthogonal unit vectors).")
print("  In weight space, this means SLERP-merged weights are the same scale as the parents.")

In [ ]:
# A6 — SLERP fallback: near-parallel vectors
# When two vectors are nearly parallel, omega ~ 0 and sin(omega) ~ 0.
# Dividing by sin(omega) causes numerical instability, so slerp falls back to lerp.
import torch

# Nearly parallel vectors (small angle between them)
v0_raw = torch.tensor([1.0, 0.01, 0.0, 0.0])
v1_raw = torch.tensor([1.0, -0.01, 0.0, 0.0])

# Normalize to unit vectors
v0 = v0_raw / v0_raw.norm()
v1 = v1_raw / v1_raw.norm()

dot = (v0 * v1).sum()
omega_deg = torch.acos(dot.clamp(-1, 1)).item() * 180 / 3.14159

print(f"v0 (normalized): {[round(x, 6) for x in v0.tolist()]}")
print(f"v1 (normalized): {[round(x, 6) for x in v1.tolist()]}")
print(f"Dot product: {dot.item():.6f}")
print(f"Angle between vectors: {omega_deg:.4f} degrees")
print()

print(f"{'t':>5}  {'SLERP result':>35}  {'LERP result':>35}  {'max diff':>10}")
print("-" * 90)
for t in [0.0, 0.25, 0.5, 0.75, 1.0]:
    sv = slerp(t, v0, v1)
    lv = (1 - t) * v0 + t * v1
    sv_str = "[" + ", ".join(f"{x:.5f}" for x in sv.tolist()) + "]"
    lv_str = "[" + ", ".join(f"{x:.5f}" for x in lv.tolist()) + "]"
    max_diff = (sv - lv).abs().max().item()
    print(f"{t:>5.2f}  {sv_str:>35}  {lv_str:>35}  {max_diff:>10.8f}")

print()
print("Observation: SLERP and LERP produce nearly identical results for near-parallel vectors.")
print("The fallback (if omega < 1e-4: return lerp) is mathematically safe here.")
print("Rationale: when the angle is tiny, the sphere surface and straight line are indistinguishable.")

In [ ]:
# A7 — t-Parameter Effect on Mock Logit Outputs
# Simulates how the t parameter shifts a merged model's preference
# between token 0 (model A's preference) and token 2 (model B's preference).
#
# Note: real merging interpolates weight tensors, not logits directly.
# This demo uses logit interpolation as a simple proxy for illustration.

import torch

logits_a = torch.tensor([3.0, 1.0, 0.5])  # model A: strongly prefers token 0
logits_b = torch.tensor([0.5, 1.0, 3.0])  # model B: strongly prefers token 2

token_labels = ["token_0", "token_1", "token_2"]

print(f"Model A logits: {logits_a.tolist()} -> argmax = token_{logits_a.argmax().item()}")
print(f"Model B logits: {logits_b.tolist()} -> argmax = token_{logits_b.argmax().item()}")
print()
print(f"{'t':>5}  {'logits':>30}  {'argmax':>10}  {'interpretation':>35}")
print("-" * 85)

interps = {
    0.0: "pure model A",
    0.3: "mostly model A",
    0.5: "equal blend",
    0.7: "mostly model B",
    1.0: "pure model B",
}

for t in [0.0, 0.3, 0.5, 0.7, 1.0]:
    merged = (1 - t) * logits_a + t * logits_b
    argmax = merged.argmax().item()
    logit_str = "[" + ", ".join(f"{x:.2f}" for x in merged.tolist()) + "]"
    print(f"{t:>5.1f}  {logit_str:>30}  {token_labels[argmax]:>10}  {interps[t]:>35}")

print()
print("Key insight: t=0 reproduces model A exactly; t=1 reproduces model B exactly.")
print("Values between 0 and 1 blend the models preferences.")
print("In real weight-space merging, this shift is controlled at the parameter level, not the logit level.")

In [ ]:
# A8 — merge_state_dicts() demo with all three cases
# Case 1: shapes match, floating point  -> SLERP merged
# Case 2: shapes differ                 -> kept from A (no merge possible)
# Case 3: key only in A                 -> kept from A (B does not have it)

import torch

def slerp_inner(t, v0, v1):
    v0_f, v1_f = v0.float(), v1.float()
    dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-4:
        return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
    so = torch.sin(omega)
    return (torch.sin((1 - t) * omega) / so * v0_f + torch.sin(t * omega) / so * v1_f).to(v0.dtype)

def merge_state_dicts(sd_a: dict, sd_b: dict, t: float) -> dict:
    merged = {}
    for key in sd_a:
        if key in sd_b and sd_a[key].shape == sd_b[key].shape and sd_a[key].is_floating_point():
            merged[key] = slerp_inner(t, sd_a[key], sd_b[key])
        else:
            merged[key] = sd_a[key]
    return merged


torch.manual_seed(7)

sd_a = {
    "transformer.layer.weight": torch.randn(3, 3),
    "transformer.layer.bias":   torch.randn(3),
    "lm_head.weight":           torch.randn(4, 8),
    "embed_tokens.weight":      torch.randint(0, 10, (10, 4)).float(),
}

sd_b = {
    "transformer.layer.weight": torch.randn(3, 3),
    "transformer.layer.bias":   torch.randn(3),
    "lm_head.weight":           torch.randn(5, 8),   # shape (5,8) != (4,8)
    "embed_tokens.weight":      torch.randint(0, 10, (10, 4)).float(),
}

merged = merge_state_dicts(sd_a, sd_b, t=0.5)

print(f"{'Key':>30}  {'Status':>50}  {'First 3 values of merged'}")
print("-" * 115)

for key in sd_a:
    val = merged[key]
    flat = val.flatten()[:3].tolist()
    flat_str = "[" + ", ".join(f"{x:.4f}" for x in flat) + "]"

    in_b = key in sd_b
    shape_match = in_b and (sd_a[key].shape == sd_b[key].shape)
    is_float = sd_a[key].is_floating_point()

    if shape_match and is_float:
        status = "SLERP merged"
    elif not shape_match and in_b:
        status = f"kept from A - shape mismatch {tuple(sd_a[key].shape)} vs {tuple(sd_b[key].shape)}"
    else:
        status = "kept from A"
    print(f"{key:>30}  {status:>50}  {flat_str}")

### Exercise 1 — Find the Optimal Merge Coefficient

In practice you do not know in advance which `t` produces the best merged model. The standard approach is a sweep: try a grid of `t` values, evaluate each merged model on a validation set, and pick the `t` with the highest score.

**Problem:** Write a function `find_optimal_t(outputs_a, outputs_b, t_values)` that:

1. For each `t` in `t_values`, compute a mock "merged score" by linearly blending `outputs_a` and `outputs_b`
2. Compute the average score across all tasks
3. Return `(best_t, best_score)` — the `t` with the highest average

The real version would load two models, call `merge_state_dicts(sd_a, sd_b, t)`, load the merged weights, and run inference on validation prompts. We use pre-computed mock scores here to keep the cell CPU-only and fast.

**Hints:**
- Use `merge_outputs_mock(t, outputs_a, outputs_b)` provided in the stub to blend the scores
- `sum(scores) / len(scores)` gives the average
- Track `best_t` by comparing `avg_score > best_score` inside the loop

In [ ]:
# Exercise 1: Find optimal t (runnable mock version)
# TODO: Fill in find_optimal_t() -- it should:
#   1. For each t_value, call merge_outputs_mock(t_value, outputs_a, outputs_b)
#   2. Compute the average score across all tasks
#   3. Return (best_t, best_score)

def merge_outputs_mock(t, outputs_a, outputs_b):
    """Mock merge: interpolate numeric scores as a proxy for merged model output quality.
    In the real version this would be: load merged model weights and run inference.
    """
    # TODO: Replace this with real merged-model evaluation
    return [a * (1 - t) + b * t for a, b in zip(outputs_a, outputs_b)]

def find_optimal_t(outputs_a, outputs_b, t_values):
    """TODO: implement this function.
    Should return (best_t, best_avg_score).
    """
    pass  # replace with your implementation

# Test data
OUTPUTS_A_SCORES = [0.9, 0.3, 0.7]  # model A quality on 3 tasks
OUTPUTS_B_SCORES = [0.4, 0.8, 0.6]  # model B quality on 3 tasks
# Best blend: want high on all 3 tasks
T_VALUES = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

result = find_optimal_t(OUTPUTS_A_SCORES, OUTPUTS_B_SCORES, T_VALUES)
print(f"Optimal t: {result}")

In [ ]:
# Exercise 1 -- Answer Key
# Key insight: the optimal t balances model A's strengths on some tasks
# against model B's strengths on others. Neither pure A nor pure B is optimal here.

def merge_outputs_mock(t, outputs_a, outputs_b):
    return [a * (1 - t) + b * t for a, b in zip(outputs_a, outputs_b)]

def find_optimal_t_answer(outputs_a, outputs_b, t_values):
    best_t, best_score = None, -1.0
    for t in t_values:
        merged_scores = merge_outputs_mock(t, outputs_a, outputs_b)
        avg_score = sum(merged_scores) / len(merged_scores)
        if avg_score > best_score:
            best_score = avg_score
            best_t = t
    return best_t, best_score

OUTPUTS_A_SCORES = [0.9, 0.3, 0.7]
OUTPUTS_B_SCORES = [0.4, 0.8, 0.6]
T_VALUES = [0.0, 0.2, 0.4, 0.5, 0.6, 0.8, 1.0]

optimal_t, score = find_optimal_t_answer(OUTPUTS_A_SCORES, OUTPUTS_B_SCORES, T_VALUES)
print(f"Optimal t = {optimal_t}  (average score = {score:.3f})")
print()
print("Score breakdown at each t:")
for t in T_VALUES:
    scores = merge_outputs_mock(t, OUTPUTS_A_SCORES, OUTPUTS_B_SCORES)
    avg = sum(scores) / len(scores)
    marker = "  <-- optimal" if t == optimal_t else ""
    print(f"  t={t}  scores={[round(s, 2) for s in scores]}  avg={avg:.3f}{marker}")

### Exercise 2 — Skip Divergent Layers

Not all layers benefit equally from merging. If two models have learned very different representations for a layer (e.g., opposite directions in weight space), merging those layers can hurt more than help.

**Problem:** Write `merge_state_dicts_selective(sd_a, sd_b, t, min_cosine_sim=0.3)` that:

1. For each key in `sd_a`, check if the key exists in `sd_b`, shapes match, and the tensor is floating-point
2. Compute the cosine similarity between `sd_a[key].flatten()` and `sd_b[key].flatten()`
3. If `cosine_sim >= min_cosine_sim`: SLERP merge as normal
4. If `cosine_sim < min_cosine_sim`: keep `sd_a[key]` unchanged (skip the merge for this layer)

**Hint:** `torch.nn.functional.cosine_similarity(a.flatten().unsqueeze(0), b.flatten().unsqueeze(0))` returns a scalar tensor with the cosine similarity.

In [ ]:
import torch
import torch.nn.functional as F

# Exercise 2: Selective merging -- skip layers where models diverge too much
# TODO: Fill in the cosine similarity check inside merge_state_dicts_selective

def merge_state_dicts_selective(sd_a, sd_b, t, min_cosine_sim=0.3):
    """SLERP merge, but skip layers where cosine similarity < min_cosine_sim.

    Args:
        sd_a: state dict of model A (kept as base when skipping)
        sd_b: state dict of model B
        t: SLERP interpolation parameter
        min_cosine_sim: layers with cosine similarity below this threshold are kept from A

    Returns:
        Merged state dict.
    """
    # TODO: for each key in sd_a:
    #   1. Check if key is in sd_b and shapes match and is_floating_point
    #   2. Compute cosine similarity between sd_a[key].flatten() and sd_b[key].flatten()
    #      using torch.nn.functional.cosine_similarity with unsqueeze(0)
    #   3. If cosine_sim >= min_cosine_sim: slerp merge
    #   4. Else: keep sd_a[key]
    pass

# Test data
torch.manual_seed(42)
sd_a = {
    "layer1.w": torch.randn(3, 3),
    "layer2.w": torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]),
}
sd_b = {
    "layer1.w": torch.randn(3, 3),
    # layer2.w in model B is the exact negative of model A -- cosine_sim = -1.0
    "layer2.w": torch.tensor([[-1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, -1.0]]),
}

result = merge_state_dicts_selective(sd_a, sd_b, t=0.5, min_cosine_sim=0.3)
print("Result:", result)

In [ ]:
# Exercise 2 -- Answer Key

import torch
import torch.nn.functional as F

def slerp_local(t, v0, v1):
    v0_f, v1_f = v0.float(), v1.float()
    dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-4:
        return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
    so = torch.sin(omega)
    return (
        torch.sin((1 - t) * omega) / so * v0_f
        + torch.sin(t * omega) / so * v1_f
    ).to(v0.dtype)

def merge_state_dicts_selective(sd_a, sd_b, t, min_cosine_sim=0.3):
    merged = {}
    for key in sd_a:
        if key in sd_b and sd_a[key].shape == sd_b[key].shape and sd_a[key].is_floating_point():
            a_flat = sd_a[key].float().flatten().unsqueeze(0)
            b_flat = sd_b[key].float().flatten().unsqueeze(0)
            cosine_sim = F.cosine_similarity(a_flat, b_flat).item()
            if cosine_sim >= min_cosine_sim:
                merged[key] = slerp_local(t, sd_a[key], sd_b[key])
                status = f"MERGED  (cosine_sim={cosine_sim:+.3f} >= {min_cosine_sim})"
            else:
                merged[key] = sd_a[key]
                status = f"KEPT from A  (cosine_sim={cosine_sim:+.3f} < {min_cosine_sim} -- divergent)"
        else:
            merged[key] = sd_a[key]
            status = "KEPT from A  (shape mismatch or non-float)"
        print(f"  {key}: {status}")
    return merged

torch.manual_seed(42)
sd_a = {
    "layer1.w": torch.randn(3, 3),
    "layer2.w": torch.tensor([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]),
}
sd_b = {
    "layer1.w": torch.randn(3, 3),
    "layer2.w": torch.tensor([[-1.0, 0.0, 0.0], [0.0, -1.0, 0.0], [0.0, 0.0, -1.0]]),
}

print("Selective merge at t=0.5, min_cosine_sim=0.3:")
result = merge_state_dicts_selective(sd_a, sd_b, t=0.5, min_cosine_sim=0.3)
print()
print("Explanation:")
print("  layer1.w: random tensors from the same distribution -- likely similar direction -> merged")
print("  layer2.w: identity vs negative identity -- cosine_sim = -1.0 -> skipped (kept from A)")

### A9 — Known Merge Recipes

Community practitioners have identified several reliable recipes. These are starting points; actual optimal `t` should always be validated on a task-specific evaluation set.

| Recipe | Model A | Model B | t range | Effect |
|---|---|---|---|---|
| Instruction boost | Base model (pretrained only) | Instruction-tuned variant | 0.3 - 0.5 | Adds instruction-following capability while retaining base knowledge |
| Domain + instruction | Domain specialist (e.g., medical) | General instruct model | 0.5 | Domain knowledge with conversational style |
| Skill blend | Coding specialist | Math specialist | 0.5 | Both skills present; may degrade on each task individually |
| Partial safety alignment | Fine-tuned model | Safety RLHF checkpoint | 0.2 - 0.4 | Partial safety behavior without full safety fine-tune overhead |

**Most reliable recipe:** merging two instruction-tuned models that were both fine-tuned from the **same base checkpoint**. Their weight tensors start from the same initialization and diverge only during fine-tuning, so the angle `omega` between corresponding weight tensors stays small and SLERP interpolation is well-behaved.

### A10 — What Happens Internally During a Merge

It is worth building a mental model of exactly what `merge_state_dicts` does step by step:

```
1. Iterate over every key in model A's state dict.

2. For each key:
   a. Is this key also in model B?           -> No:  keep model A value
   b. Do the tensor shapes match?            -> No:  keep model A value
   c. Is the tensor floating-point dtype?    -> No:  keep model A value (skip integer masks, etc.)
   d. All checks pass:                       -> SLERP(t, param_A, param_B)

3. The result is a new state dict with the same keys as model A.
   Load it into a model instance with model.load_state_dict(merged_sd).
```

**Important:** the merge does NOT:
- Retrain any weights
- Require gradients
- Need a GPU (CPU is fine; it is just tensor arithmetic)
- Change the model architecture (same config, same number of parameters)

The merged model can be saved with `model.save_pretrained(...)` and shared on HuggingFace Hub just like any other model.

In [ ]:
# A11 -- Layer coverage analysis (no model download needed)
# Demonstrates how to predict what fraction of layers will actually merge
# for two models with different hidden dimensions.

# Mock architecture configs for SmolLM2-135M and SmolLM2-360M
# Real shapes approximate the actual SmolLM2 architectures
ARCH_135M = {
    "model.embed_tokens.weight":                    (49152, 576),
    "model.layers.0.self_attn.q_proj.weight":       (576, 576),
    "model.layers.0.self_attn.k_proj.weight":       (192, 576),
    "model.layers.0.self_attn.v_proj.weight":       (192, 576),
    "model.layers.0.self_attn.o_proj.weight":       (576, 576),
    "model.layers.0.mlp.gate_proj.weight":          (1536, 576),
    "model.layers.0.mlp.up_proj.weight":            (1536, 576),
    "model.layers.0.mlp.down_proj.weight":          (576, 1536),
    "model.layers.0.input_layernorm.weight":        (576,),
    "lm_head.weight":                               (49152, 576),
}

ARCH_360M = {
    "model.embed_tokens.weight":                    (49152, 960),
    "model.layers.0.self_attn.q_proj.weight":       (960, 960),
    "model.layers.0.self_attn.k_proj.weight":       (320, 960),
    "model.layers.0.self_attn.v_proj.weight":       (320, 960),
    "model.layers.0.self_attn.o_proj.weight":       (960, 960),
    "model.layers.0.mlp.gate_proj.weight":          (2560, 960),
    "model.layers.0.mlp.up_proj.weight":            (2560, 960),
    "model.layers.0.mlp.down_proj.weight":          (960, 2560),
    "model.layers.0.input_layernorm.weight":        (960,),
    "lm_head.weight":                               (49152, 960),
}

print(f"{'Key':>48}  {'135M shape':>15}  {'360M shape':>15}  {'Will merge?':>12}")
print("-" * 98)

n_merge = 0
n_total = 0
for key in ARCH_135M:
    shape_a = ARCH_135M[key]
    shape_b = ARCH_360M.get(key, "missing")
    will_merge = (shape_b != "missing") and (shape_a == shape_b)
    if will_merge:
        n_merge += 1
    n_total += 1
    shape_b_str = str(shape_b) if shape_b != "missing" else "missing"
    print(f"{key:>48}  {str(shape_a):>15}  {shape_b_str:>15}  {'YES' if will_merge else 'no':>12}")

print()
print(f"Summary (one layer sample): {n_merge}/{n_total} would merge ({100*n_merge/n_total:.0f}%)")
print("Across the full model: only layernorm weights with shape (hidden_dim,) might share shapes.")
print("135M hidden_dim=576, 360M hidden_dim=960 -- these differ, so nothing merges.")

### A12 — t Sweep Visualization (ASCII)

Below is a schematic of what a t-sweep evaluation loop looks like in practice. Each merged model is a distinct point on the arc between model A and model B:

```
Weight space (simplified 2D projection)

  Model A                                    Model B
     *                                          *
      \                                        /
       \     t=0.2     t=0.5     t=0.8        /
        +------*---------*---------*----------+
               |         |         |
             eval       eval      eval
            score=0.71 score=0.74 score=0.69
                            ^
                       optimal t
```

**Practical sweep recipe:**

```python
T_GRID = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

results = []
for t in T_GRID:
    merged_sd = merge_state_dicts(sd_a, sd_b, t)
    model_a.load_state_dict(merged_sd)
    score = evaluate(model_a, val_prompts, val_answers)
    results.append((t, score))

best_t, best_score = max(results, key=lambda x: x[1])
```

For large models (7B+), each `load_state_dict` call can take minutes. Use a coarser grid (0.0, 0.25, 0.5, 0.75, 1.0) first, then refine around the best region.

## Part B — Full Model Merge (SmolLM2)

**Requirements:**
- Approximately 2 GB free RAM (two SmolLM2 models loaded simultaneously in float32 on CPU)
- No GPU required
- Internet connection for first download (approximately 300 MB total)

**Why SmolLM2-135M + SmolLM2-360M?**

These two models share the same vocabulary (49,152 tokens) and are both instruction-tuned by the same team (HuggingFaceTB). However, their hidden dimensions differ (576 vs 960), so most projection layers will NOT merge (shape mismatch). The layers that will SLERP-merge are only those that happen to share identical shapes — in this cross-size pairing, that is very few.

This means the Part B merge is primarily a demonstration of the mechanics and a comparison of outputs, not a high-quality capability blend. For a quality blend, merge two models of the **same size** (e.g., two SmolLM2-135M fine-tunes from the same base).

In [ ]:
# B1 -- RAM check before loading models
import psutil

available_gb = psutil.virtual_memory().available / (1024 ** 3)
required_gb = 2.0

print(f"Available RAM: {available_gb:.1f} GB")
if available_gb < required_gb:
    raise RuntimeError(
        f"Need at least {required_gb} GB free RAM, only {available_gb:.1f} GB available.\n"
        "Close other applications and retry, or use a smaller model pair."
    )
print("RAM check passed -- proceeding to load models.")

In [ ]:
# B2 -- Load models, merge, compare (requires ~2 GB RAM)
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# -- inline helpers so cell is self-contained --
def slerp(t, v0, v1):
    v0_f, v1_f = v0.float(), v1.float()
    dot = (v0_f * v1_f).sum() / (v0_f.norm() * v1_f.norm() + 1e-8)
    dot = dot.clamp(-1, 1)
    omega = torch.acos(dot)
    if omega.abs() < 1e-4:
        return ((1 - t) * v0_f + t * v1_f).to(v0.dtype)
    so = torch.sin(omega)
    return (
        torch.sin((1 - t) * omega) / so * v0_f
        + torch.sin(t * omega) / so * v1_f
    ).to(v0.dtype)

def merge_state_dicts(sd_a, sd_b, t):
    merged = {}
    for key in sd_a:
        if key in sd_b and sd_a[key].shape == sd_b[key].shape and sd_a[key].is_floating_point():
            merged[key] = slerp(t, sd_a[key], sd_b[key])
        else:
            merged[key] = sd_a[key]
    return merged

def generate(model, tokenizer, prompt, max_new_tokens=64):
    inputs = tokenizer(prompt, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    ).strip()


MODEL_A = "HuggingFaceTB/SmolLM2-135M-Instruct"
MODEL_B = "HuggingFaceTB/SmolLM2-360M-Instruct"
T = 0.5

EVAL_PROMPTS = [
    "What is the capital of France?",
    "Explain gradient descent in one sentence.",
    "Write a haiku about Python.",
]

print(f"Loading model A: {MODEL_A}")
tok_a = AutoTokenizer.from_pretrained(MODEL_A)
model_a = AutoModelForCausalLM.from_pretrained(MODEL_A)

print(f"Loading model B: {MODEL_B}")
tok_b = AutoTokenizer.from_pretrained(MODEL_B)
model_b = AutoModelForCausalLM.from_pretrained(MODEL_B)

print("Running eval on model A and model B ...")
outputs_a = [generate(model_a, tok_a, p) for p in EVAL_PROMPTS]
outputs_b = [generate(model_b, tok_b, p) for p in EVAL_PROMPTS]

print(f"\nMerging state dicts at t={T} ...")
sd_a = model_a.state_dict()
sd_b = model_b.state_dict()
merged_sd = merge_state_dicts(sd_a, sd_b, T)

n_merged = sum(
    1 for k in sd_a
    if k in sd_b
    and sd_a[k].shape == sd_b[k].shape
    and sd_a[k].is_floating_point()
)
n_total = len(sd_a)
print(f"  {n_merged}/{n_total} layers SLERP merged; {n_total - n_merged} kept from model A")

model_a.load_state_dict(merged_sd)
outputs_merged = [generate(model_a, tok_a, p) for p in EVAL_PROMPTS]

print("\n" + "=" * 70)
for i, prompt in enumerate(EVAL_PROMPTS):
    print(f"\nPrompt: {prompt}")
    print(f"  Model A : {outputs_a[i][:100]}")
    print(f"  Model B : {outputs_b[i][:100]}")
    print(f"  Merged  : {outputs_merged[i][:100]}")

### B3 — Interpreting the Merge Results

Because SmolLM2-135M and SmolLM2-360M have different hidden dimensions, most weight tensors did not SLERP-merge (shape mismatch). The merged model is therefore very close to model A with only a small number of shared-shape layers blended.

**What to look for in the output table:**

- If merged output closely resembles model A: the merge ratio was low (few layers merged) — expected for cross-size merges
- If merged output is noticeably different from both A and B: some key layers did merge and shifted the behavior
- If merged output produces garbled text: the merged layers introduced incoherence — try a lower `t` (closer to model A)

**For a high-quality merge:** use two fine-tuned versions of the same base checkpoint at the same parameter count. For example, merge SmolLM2-135M-Instruct with a SmolLM2-135M fine-tuned on coding tasks — nearly all layers will merge because all shapes are identical.

### B4 — Try It: Different t Values

Re-run the merge cell above with different `T` values to observe the effect:

| T value | Expected behavior |
|---|---|
| 0.0 | Pure model A output |
| 0.1 | Mostly model A, slight influence from B |
| 0.5 | Equal blend (what we ran above) |
| 0.9 | Mostly model B weights; output may degrade for cross-size merges |
| 1.0 | Model A architecture loaded with model B's shared-shape weights |

Note that `T=1.0` does NOT produce model B output — it produces model A's architecture with B's values in the few matching layers. To get pure model B output, you would need to load model B directly.

**Extension challenge:** modify the code above to run a t-sweep over `[0.0, 0.25, 0.5, 0.75, 1.0]` and print all outputs in a table. Which `t` gives the most coherent responses?

## Summary — Key Takeaways

**SLERP mechanics:**
- LERP travels through the interior of the weight-space sphere and loses magnitude at midpoints
- SLERP travels the arc on the sphere surface, preserving vector magnitude throughout interpolation
- When omega < 1e-4 (nearly parallel vectors), SLERP falls back to LERP safely

**State dict merge rules:**
1. Key must exist in both models
2. Tensor shapes must match exactly
3. Tensor must be floating-point dtype
4. Failure on any check: keep model A's value unchanged

**Practical guidance:**
- Best results come from merging two models fine-tuned from the same base checkpoint at the same size
- Always check `n_merged / n_total` — low merge coverage means the merge is essentially model A
- Use a t-sweep with validation evaluation to find the optimal blend coefficient
- Consider selective merging (skip layers with low cosine similarity) for more reliable merges
- Merging is free in compute terms but does not guarantee quality — always evaluate before deploying

## What's Next

- **SLERP foundation:** "Merging Models with Fisher-Weighted Averaging" (Matena & Raffel, 2022) is the foundational work on weight-space merging and explains the theoretical basis for why interpolation in weight space can combine capabilities
- **MergeKit:** open-source tool at `github.com/arcee-ai/mergekit` supporting SLERP, TIES, DARE, and more -- handles config-driven merges, quantization, and large model sharding for models that do not fit in RAM
- **TIES-merging:** "Resolving Interference When Merging Models" (Yadav et al., 2023) addresses sign conflicts that arise when merging many fine-tuned models simultaneously -- more robust than SLERP for multi-model merges
- **DARE:** "Language Models are Super Mario" (Yu et al., 2023) randomly drops small delta weights before merging to reduce task interference -- particularly effective for large model families with many fine-tunes
- **Previous in this repo:** `examples/146-orpo-alignment/` -- train a model with ORPO to prefer outputs; once trained, that model is a candidate for merging with a domain specialist